In [1]:
import pandas as pd
import numpy as np
import sklearn

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

print("Training environment is ready.")
print("scikit-learn version:", sklearn.__version__)

Training environment is ready.
scikit-learn version: 1.6.1


In [2]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

print("NeuroTwin training environment is ready.")

NeuroTwin training environment is ready.


In [3]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 180

# Synthetic examples for testing code only.
sleep = np.clip(rng.normal(7.2, 0.9, n), 4, 10)
activity = np.clip(rng.normal(35, 15, n), 0, 120)
stress = rng.integers(1, 10, n)
previous_reaction = np.clip(rng.normal(330, 35, n), 180, 600)

# Artificial relationship—not a scientifically established formula.
next_reaction = (
    330
    + 0.35 * (previous_reaction - 330)
    - 8 * (sleep - 7)
    + 4 * (stress - 5)
    + rng.normal(0, 22, n)
)

data = pd.DataFrame({
    "sleep": sleep,
    "activity": activity,
    "stress": stress,
    "previous_reaction_ms": previous_reaction,
    "target_reaction_ms": next_reaction,
})

print("SYNTHETIC DATA — pipeline testing only")
print("Rows:", len(data))
display(data.head())

SYNTHETIC DATA — pipeline testing only
Rows: 180


,sleep,activity,stress,previous_reaction_ms,target_reaction_ms
0,7.474245,54.590026,3,278.309976,288.378577
1,6.264014,38.290738,2,367.352785,327.905317
2,7.875406,28.836092,9,318.397914,364.914159
3,8.046508,51.594331,2,369.010736,300.388798
4,5.444068,41.431347,1,343.418199,278.062656


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

features = [
    "sleep",
    "activity",
    "stress",
    "previous_reaction_ms",
]

X = data[features]
y = data["target_reaction_ms"]

# Keep the final 20% separate from training.
# These synthetic rows are independent, not real longitudinal history.
split = int(len(data) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]

# Scaling is learned from the training portion only.
model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0)),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

# Simple comparison: predict the previous reaction result.
baseline_predictions = X_test["previous_reaction_ms"]

model_mae = mean_absolute_error(y_test, predictions)
baseline_mae = mean_absolute_error(y_test, baseline_predictions)

print("SYNTHETIC EXPERIMENT — not real-world validation")
print(f"Training examples: {len(X_train)}")
print(f"Test examples: {len(X_test)}")
print(f"Model MAE: {model_mae:.2f} ms")
print(f"Previous-result baseline MAE: {baseline_mae:.2f} ms")

comparison = pd.DataFrame({
    "Actual synthetic result (ms)": y_test.to_numpy(),
    "Model prediction (ms)": predictions,
    "Baseline prediction (ms)": baseline_predictions.to_numpy(),
})

display(comparison.head(10).round(2))

SYNTHETIC EXPERIMENT — not real-world validation
Training examples: 144
Test examples: 36
Model MAE: 17.63 ms
Previous-result baseline MAE: 21.93 ms


,Actual synthetic result (ms),Model prediction (ms),Baseline prediction (ms)
0,390.99,347.27,386.39
1,370.45,341.18,287.22
2,373.76,342.67,338.72
3,361.34,348.10,393.75
4,330.20,304.07,272.19
5,295.84,292.99,285.16
6,312.81,303.91,315.17
7,309.42,335.56,311.78
8,343.15,323.83,358.44
9,320.19,310.11,338.46


In [6]:
import json
from google.colab import files

scaler = model.named_steps["scaler"]
ridge = model.named_steps["ridge"]

artifact = {
    "version": "synthetic-demo-v1",
    "training_data": "synthetic",
    "purpose": "Pipeline demonstration only",
    "target": "reaction_ms",
    "features": features,
    "means": scaler.mean_.tolist(),
    "scales": scaler.scale_.tolist(),
    "coefficients": ridge.coef_.tolist(),
    "intercept": float(ridge.intercept_),
    "evaluation": {
        "training_examples": len(X_train),
        "test_examples": len(X_test),
        "model_mae_ms": float(model_mae),
        "baseline_mae_ms": float(baseline_mae),
    },
    "real_world_validated": False,
}

# Verify that the exported values reproduce Python predictions.
exported_predictions = (
    ((X_test.to_numpy() - scaler.mean_) / scaler.scale_)
    @ ridge.coef_
    + ridge.intercept_
)

assert np.allclose(exported_predictions, predictions)

filename = "neurotwin-synthetic-demo-model.json"

with open(filename, "w") as file:
    json.dump(artifact, file, indent=2, allow_nan=False)

print("Export verification passed.")
files.download(filename)

Export verification passed.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>